In [1]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, GPT2Config
from transformers import get_linear_schedule_with_warmup

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split, RandomSampler, SequentialSampler

import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name: ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']
model_name = "gpt2-large" 
model_save_path = './model'

In [2]:
import datasets
train = datasets.load_from_disk('/u1/kfountou/positional_attention/data/processed/bank_minimum_complete/train')
val = datasets.load_from_disk('/u1/kfountou/positional_attention/data/processed/bank_minimum_complete/val').select(range(10))
test = datasets.load_from_disk('/u1/kfountou/positional_attention/data/processed/bank_minimum_complete/test').select(range(10))

In [3]:
train.shape

(500000, 4)

In [4]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)

tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

model = model.to(device)

In [5]:
train['serialized'][0]

'Entry 1. The type of job is services. The education is secondary. The average yearly balance, in euros is 0.56326.\nEntry 2. The type of job is admin.. The education is secondary. The average yearly balance, in euros is 1.83869.\nEntry 3. The type of job is entrepreneur. The education is secondary. The average yearly balance, in euros is 1.38123.\nEntry 4. The type of job is blue-collar. The education is secondary. The average yearly balance, in euros is 0.98099.\n Minimum of balance where job is entrepreneur? Answer: '

In [6]:
train['answer'][0]

'1.38123'

In [7]:
from nlp_dataset import generate_sample

def form_string(sample: tuple[list, float], isTrain: bool) ->tuple[str, float]:
    sample_text = sample[0]
    sample_ans = sample[1]
    # input_list = [x if type(x) == str else format(x, '05.2f') for x in sample_text[:-1]]
    prompt = "<|startoftext|>" + sample_text
    if isTrain:
        prompt += sample_ans
        prompt += "<|endoftext|>"
    # else:
    #     prompt += " Answer: "
    return prompt, float(sample_ans)

In [8]:
# num_cats = 8
# query_type = "min"
# num_query_cats = 4
# train_low = 0
# train_high = 5
# test_low = 0
# test_high = 20
# num_train_samples = 50000
# num_test_samples = 100
# num_val_samples = 100
# num_final_test_samples = 1000

In [9]:
train_data_comb = [form_string( (x['serialized'], x['answer']), True) for x in train]
train_data = [x[0] for x in train_data_comb]
train_data_ans = [x[1] for x in train_data_comb]

test_data_comb = [form_string( (x['serialized'], x['answer']), False) for x in test]
test_data = [x[0] for x in test_data_comb]
test_data_ans = [x[1] for x in test_data_comb]

val_data_comb = [form_string( (x['serialized'], x['answer']), False) for x in val]
val_data = [x[0] for x in val_data_comb]
val_data_ans = [x[1] for x in val_data_comb]

# test_data_all, test_data_all_ans = [], []
# for i in range(1, 11):
#     cur = [form_string(generate_sample(num_cats, query_type, 0, 5 * i, num_query_cats, train=False), False) for _ in range(num_final_test_samples)]
#     test_data_all.append([x[0] for x in cur])
#     test_data_all_ans.append([x[1] for x in cur])


In [10]:
print(val_data[0])
print(val_data_ans[0])

<|startoftext|>Entry 1. The type of job is management. The education is tertiary. The average yearly balance, in euros is 0.20279.
Entry 2. The type of job is blue-collar. The education is primary. The average yearly balance, in euros is 0.48242.
Entry 3. The type of job is management. The education is tertiary. The average yearly balance, in euros is 1.77815.
Entry 4. The type of job is blue-collar. The education is primary. The average yearly balance, in euros is 0.84282.
 Minimum of balance where job is blue-collar? Answer: 
0.48242


In [11]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name,
                                              bos_token='<|startoftext|>',
                                              eos_token='<|endoftext|>',
                                              unk_token='<|unknown|>',
                                              pad_token='<|pad|>'
                                             )

In [12]:
batch_size = 32
max_length = 150

# standard PyTorch approach of loading data in using a Dataset class.
class NAR_Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.input_ids = []
        self.attn_masks = []
        self.labels = []

        for data_point in data:
            encodings = tokenizer.encode_plus(data_point,
                                              truncation=True,
                                              padding='max_length',
                                              max_length=max_length,
                                              # return a PyTorch tensor
                                              return_tensors='pt'       
                                             )
            input_ids = torch.squeeze(encodings['input_ids'],0)
            self.input_ids.append(input_ids)
            self.attn_masks.append(torch.squeeze(encodings['attention_mask'],0))

            idx = (input_ids == 23998).nonzero(as_tuple=True)[0]
            labels = torch.tensor(input_ids)
            labels[0:idx] = -100
            self.labels.append(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_masks[idx], self.labels[idx]

dataset_indist_train = NAR_Dataset(train_data, tokenizer)
dataset_indist_val = NAR_Dataset(val_data, tokenizer)
dataset_ood = NAR_Dataset(test_data, tokenizer)
print(f"input_ids: {dataset_ood[0][0]} attn_masks: {dataset_ood[0][1]} labels: {dataset_ood[0][2]}")

/tmp/ipykernel_1277884/4213365263.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(input_ids)


input_ids: tensor([50257, 30150,   352,    13,   383,  2099,   286,  1693,   318,  3710,
           13,   383,  3707,   318,  9233,    13,   383,  2811, 24169,  5236,
           11,   287, 21138,   318,   604,    13,  1157, 21601,    13,   198,
        30150,   362,    13,   383,  2099,   286,  1693,   318,  9880,    13,
          383,  3707,   318,  4165,    13,   383,  2811, 24169,  5236,    11,
          287, 21138,   318,   807,    13, 33300,  1415,    13,   198, 30150,
          513,    13,   383,  2099,   286,  1693,   318, 13169,   492,   383,
         3707,   318,  9233,    13,   383,  2811, 24169,  5236,    11,   287,
        21138,   318,   362,    13,    15,  2996,    13,   198, 30150,   604,
           13,   383,  2099,   286,  1693,   318,  9880,    13,   383,  3707,
          318,  9233,    13,   383,  2811, 24169,  5236,    11,   287, 21138,
          318,   642,    13, 14656,  2598,    13,   198, 26265,   286,  5236,
          810,  1693,   318,  9880,    30, 23998,    

In [13]:
print(tokenizer.decode(dataset_indist_train[0][0]))

<|startoftext|>Entry 1. The type of job is services. The education is secondary. The average yearly balance, in euros is 0.56326.
Entry 2. The type of job is admin.. The education is secondary. The average yearly balance, in euros is 1.83869.
Entry 3. The type of job is entrepreneur. The education is secondary. The average yearly balance, in euros is 1.38123.
Entry 4. The type of job is blue-collar. The education is secondary. The average yearly balance, in euros is 0.98099.
 Minimum of balance where job is entrepreneur? Answer: 1.38123<|endoftext|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|>


In [14]:
print(tokenizer.decode(dataset_indist_train[99][0]))

<|startoftext|>Entry 1. The type of job is technician. The education is primary. The average yearly balance, in euros is 1.41819.
Entry 2. The type of job is management. The education is tertiary. The average yearly balance, in euros is 1.2312.
Entry 3. The type of job is blue-collar. The education is secondary. The average yearly balance, in euros is 1.77866.
Entry 4. The type of job is management. The education is tertiary. The average yearly balance, in euros is 0.57993.
 Minimum of balance where job is technician? Answer: 1.41819<|endoftext|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|>


In [15]:

train_dataloader = DataLoader(
            dataset_indist_train, 
            sampler = RandomSampler(dataset_indist_train),
            batch_size = batch_size # Trains with this batch size.
        )

# Get valiation samples sequentially.
validation_dataloader = DataLoader(
            dataset_indist_val, 
            sampler = SequentialSampler(dataset_indist_val),
            batch_size = batch_size # Evaluate with this batch size.
        )

test_dataloader = DataLoader(
            dataset_ood, 
            sampler = SequentialSampler(dataset_ood),
            batch_size = batch_size # Evaluate with this batch size.
        )
            


In [16]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)
model = model.to(device)
model.resize_token_embeddings(len(tokenizer))

epochs = 3
learning_rate = 1e-5
warmup_steps = 1e2
# to prevent any division by zero in the implementation
epsilon = 1e-8
optim = AdamW(model.parameters(), lr = learning_rate, eps = epsilon)

total_steps = len(train_dataloader) * epochs  # [no batches] x [no epochs]

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optim,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [17]:
def infer(prompt, model):
    input = prompt
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=7,
                            do_sample = True, top_k = 1, top_p = 0.85, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    start_index = output.find("Answer: ") + len("Answer: ")
    # Extract the first 5 characters from that point
    result = output[start_index:start_index + 7]

    return result

In [18]:
def infer2(prompt, model):
    input = prompt
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=7,
                            do_sample = True, top_k = 1, top_p = 0.85, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0], skip_special_tokens=True)

    return output

In [19]:
import numpy as np

def get_metrics(predictions, target):
    diff_mse, off = [], 0
    diff_mape = []
    for (i, pred) in enumerate(predictions):
        try:
            diff_mse.append(abs(float(pred) - target[i])**2)
            diff_mape.append(abs(float(pred) - ((abs(target[i])) + 1.0e-13)) / (abs(target[i])) + 1.0e-13)
        except:
            off += 1
    
    if len(diff_mse) == 0:
        return np.inf, np.inf, off / len(predictions) * 100
    
    return sum(diff_mse)/len(diff_mse), round(sum(diff_mape)/len(diff_mape) * 100, 2), round(off / len(predictions) * 100, 2)
    

In [21]:
done = False
for epoch_i in range(0, epochs):
    total_train_loss = 0
    model.train() 

    for step, batch in enumerate(train_dataloader): 
        b_input_ids = batch[0].to(device) 
        b_labels    = batch[2].to(device)
        b_masks     = batch[1].to(device) 

        model.zero_grad()
        outputs = model( input_ids = b_input_ids, labels = b_labels,
                         attention_mask = b_masks, token_type_ids = None )

        loss = outputs[0]

        # Get sample every x batches.
        if step % 100 == 0 and not step == 0:
            print(f"Partial Train Loss: {loss}")
            model.eval()
            test_preds = [infer(test_data[i], model) for i in range(len(test_data))]
            test_metrics = get_metrics(test_preds, test_data_ans)
            print(f"Test MSE Loss: {test_metrics[0]}, Test MAPE Loss: {test_metrics[1]}, Test Off: {test_metrics[2]}")
            val_preds = [infer(val_data[i], model) for i in range(len(val_data))]
            val_metrics = get_metrics(val_preds, val_data_ans)
            print(f"Val MSE Loss: {val_metrics[0]}, Val MAPE Loss: {val_metrics[1]}, Val Off: {val_metrics[2]}")
            train_preds = [infer(x, model) for x in train_data[1:10]]
            train_metrics = get_metrics(train_preds, train_data_ans[1:10])
            print(f"Train MSE Loss: {train_metrics[0]}, Train MAPE Loss: {train_metrics[1]}, Train Off: {train_metrics[2]}")
            print(infer2(val_data[5], model))
            # if val_metrics[0] < 0.05:
            #     done = True
            #     break
            model.train()

        if done:
            break
        loss.backward()
        optim.step()
        scheduler.step()

Partial Train Loss: 3.6343092918395996
Test MSE Loss: 0.25790433148000014, Test MAPE Loss: 3.21, Test Off: 0.0
Val MSE Loss: 7.125000000002038e-09, Val MAPE Loss: 0.01, Val Off: 20.0
Train MSE Loss: 0.0, Train MAPE Loss: 0.0, Train Off: 11.11
Entry 1. The type of job is blue-collar. The education is secondary. The average yearly balance, in euros is 0.62808.
Entry 2. The type of job is services. The education is secondary. The average yearly balance, in euros is 0.58236.
Entry 3. The type of job is admin.. The education is secondary. The average yearly balance, in euros is 0.43277.
Entry 4. The type of job is technician. The education is secondary. The average yearly balance, in euros is 1.52504.
 Minimum of balance where job is blue-collar? Answer: рубран
Partial Train Loss: 0.6354269981384277
Test MSE Loss: 0.3684347580285716, Test MAPE Loss: 4.58, Test Off: 30.0
Val MSE Loss: 0.073000207875, Val MAPE Loss: 12.51, Val Off: 20.0
Train MSE Loss: 0.0, Train MAPE Loss: 0.0, Train Off: 0.

KeyboardInterrupt: 

In [ ]:
# import os

# filename_mape = "./losses_mape_min.txt"
# filename_mse = "./losses_mse_min.txt"
# filename_off = "./losses_off_min.txt"
# os.makedirs(os.path.dirname(filename_mape), exist_ok=True) 
# os.makedirs(os.path.dirname(filename_mse), exist_ok=True)
# os.makedirs(os.path.dirname(filename_off), exist_ok=True)

# model.eval()
# for i in range(10):
#     test_preds = [infer(test_data_all[i][j]) for j in range(len(test_data_all[i]))]
#     test_metrics = get_metrics(test_preds, test_data_all_ans[i])
#     print(f"Test MSE Loss: {test_metrics[0]}, Test MAPE Loss: {test_metrics[1]}, Test Off: {test_metrics[2]}")
#     with open(filename_mape, "a") as f:
#         f.write(f"{test_metrics[1]}\t")
#     with open(filename_mse, "a") as f:
#         f.write(f"{test_metrics[0]}\t")
#     with open(filename_off, "a") as f:
#         f.write(f"{test_metrics[2]}\t")

# with open(filename_mape, "a") as f:
#     f.write("\n")
# with open(filename_mse, "a") as f:
#     f.write("\n")
# with open(filename_off, "a") as f:
#     f.write("\n")

Test MSE Loss: 0.05817420000000002, Test MAPE Loss: 5.27, Test Off: 0.1
Test MSE Loss: 0.6895070210631903, Test MAPE Loss: 7.48, Test Off: 0.3
Test MSE Loss: 1.9939122080679377, Test MAPE Loss: 10.08, Test Off: 5.8
Test MSE Loss: 4.49651417748918, Test MAPE Loss: 12.53, Test Off: 7.6
Test MSE Loss: 6.909439187705824, Test MAPE Loss: 14.12, Test Off: 8.9
Test MSE Loss: 11.91187258064516, Test MAPE Loss: 17.54, Test Off: 13.2
Test MSE Loss: 13.397812838633692, Test MAPE Loss: 17.22, Test Off: 15.1
Test MSE Loss: 13.824066707466338, Test MAPE Loss: 17.51, Test Off: 18.3
Test MSE Loss: 18.317881528662415, Test MAPE Loss: 18.15, Test Off: 21.5
Test MSE Loss: 16.660253974358977, Test MAPE Loss: 17.26, Test Off: 22.0


: 

: 

In [29]:
for step, batch in enumerate(train_dataloader): 
    b_input_ids = batch[0].to(device) 
    b_labels    = batch[2].to(device)
    b_masks     = batch[1].to(device) 
    break

In [30]:
b_masks[1,:]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0], device='cuda:0')

In [31]:
b_input_ids[1,:]

tensor([50257, 30150,   352,    13,   383,  2099,   286,  1693,   318, 13169,
          492,   383,  3707,   318,  9233,    13,   383,  2811, 24169,  5236,
           11,   287, 21138,   318,   352,    13,    20,  3132,    13,   198,
        30150,   362,    13,   383,  2099,   286,  1693,   318,  4542,    13,
          383,  3707,   318, 48358,  8042,    13,   383,  2811, 24169,  5236,
           11,   287, 21138,   318,   352,    13, 43284,  6420,    13,   198,
        30150,   513,    13,   383,  2099,   286,  1693,   318, 33024,    13,
          383,  3707,   318, 48358,  8042,    13,   383,  2811, 24169,  5236,
           11,   287, 21138,   318,   657,    13,  2791,  4846,    13,   198,
        30150,   604,    13,   383,  2099,   286,  1693,   318,  4542,    13,
          383,  3707,   318, 48358,  8042,    13,   383,  2811, 24169,  5236,
           11,   287, 21138,   318,   657,    13,    23,  2857,  4790,    13,
          198, 26265,   286,  5236,   810,  1693,   318, 13169, 

In [32]:
b_labels[1,:]

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 

In [38]:
tokenizer.decode(b_labels[1,129:], skip_special_tokens=False)

' Answer: 1.531<|endoftext|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|>'